# Latin-Masking Pipeline (Two-Stage API)

This notebook runs the latin-masking pipeline using the `run_pipeline_stage1` and
`run_pipeline_stage2` API functions, which handle normalization, sentence splitting,
UDPipe POS tagging, adverb collection, -que splitting, and masking automatically.

## Pipeline Stages:
- **Stage 1**: Normalize → sentence-split → UDPipe → collect adverbs → save `common_adverbs.txt`
- **Review**: Manually review/edit `common_adverbs.txt` before proceeding
- **Stage 2**: -que split → UDPipe → mask

## Step 1: Setup and Imports

In [14]:
from pathlib import Path

from latin_masking import run_pipeline_stage1, run_pipeline_stage2
from latin_masking.types import MaskingConfig


# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for folder in [current, *current.parents]:
        if (
            (folder / "data").is_dir()
            and (folder / "common_adverbs.txt").is_file()
        ):
            return folder

    raise FileNotFoundError("Project root not found.")


BASE_DIR = find_project_root()

INPUT_DIR = BASE_DIR / "data"

OUTPUT_DIR = BASE_DIR / "2.De operibus" / "data_processed_operibus"

MODEL = "latin-evalatin24-240520"
ADVERB_THRESHOLD = 200

EXCLUDED_FILES = {
    "X_librosregum.txt",
}

# Find raw .txt files

txt_files = sorted([
    f for f in INPUT_DIR.glob("*.txt")
    if "_sentences" not in f.name
    and f.name not in EXCLUDED_FILES
])

# Check for potential duplicates

stems = [f.stem for f in txt_files]
duplicates = [s for s in stems if stems.count(s) > 1]

if duplicates:
    print(f"\nWARNING: Found duplicate file stems: {set(duplicates)}")

# Build pipeline config

config = MaskingConfig(
    output_dir=OUTPUT_DIR,
    model=MODEL,
    adverb_threshold=ADVERB_THRESHOLD,
    common_adverbs_path=BASE_DIR / "common_adverbs.txt",
    cache_dir=OUTPUT_DIR / "udpipe_cache",
    unsafe_certs_ok=True,
)


print(f"Project root: {BASE_DIR}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

print(f"\nConfig: model={config.model}, threshold={config.adverb_threshold}")
print(f"  output_dir={config.output_dir}")
print(f"  cache_dir={config.cache_dir}")
print(f"  adverbs_path={config.common_adverbs_path}")


Config: model=latin-evalatin24-240520, threshold=200
  output_dir=data_processed_operibus
  cache_dir=data_processed_operibus/udpipe_cache
  adverbs_path=common_adverbs.txt


## Step 2: Run Stage 1 — Normalize, Split, UDPipe, Collect Adverbs

This runs the full stage 1 pipeline per file:
1. Read raw text
2. Sentence-split (with automatic preprocessing: UV/IJ normalization, macron removal, punctuation stripping)
3. UDPipe POS tagging (with caching)
4. Collect adverbs

After all files: aggregate → normalize counts → generate list → save `common_adverbs.txt`.

In [17]:
stage1_result = run_pipeline_stage1(txt_files, config=config, preserve_eol=True)

print(f"\n=== Stage 1 Summary ===")
print(f"Files processed: {len(stage1_result.sentences_per_file)}")
print(f"Total sentences: {sum(stage1_result.sentences_per_file.values())}")
print(f"Unique adverbs: {len(stage1_result.adverb_counts)}")
print(f"Adverbs saved to: {stage1_result.common_adverbs_path}")

→ AE_bhc.txt
✓ AE_bhc.txt: 399 sentences (fetched)
    68 adverbs collected
→ ASt_troilus.txt
✓ ASt_troilus.txt: 3497 sentences (fetched)
    341 adverbs collected
→ BB_adelae.txt
✓ BB_adelae.txt: 700 sentences (fetched)
    155 adverbs collected
→ BBi_speculum.txt
✓ BBi_speculum.txt: 981 sentences (fetched)
    183 adverbs collected
→ BI_reynardus.txt
✓ BI_reynardus.txt: 764 sentences (fetched)
    178 adverbs collected
→ BM_regum.txt
✓ BM_regum.txt: 610 sentences (fetched)
    101 adverbs collected
→ BS_mathematicus.txt
✓ BS_mathematicus.txt: 403 sentences (fetched)
    112 adverbs collected
→ EM_mahumeti.txt
✓ EM_mahumeti.txt: 530 sentences (fetched)
    164 adverbs collected
→ HA_pentateuchum.txt
✓ HA_pentateuchum.txt: 498 sentences (fetched)
    89 adverbs collected
→ HM_gestis.txt
✓ HM_gestis.txt: 3471 sentences (fetched)
    238 adverbs collected
→ HW_hortus.txt
✓ HW_hortus.txt: 4543 sentences (fetched)
    234 adverbs collected
→ H_carmina.txt
✓ H_carmina.txt: 297 sentences (fe

In [18]:
for path in txt_files:
    data = path.read_bytes()

    try:
        data.decode("utf-8")
    except UnicodeDecodeError as e:
        pos = e.start

        line_number = data[:pos].count(b"\n") + 1

        print("=" * 70)
        print(f"FILE: {path}")
        print(f"BYTE POSITION: {pos}")
        print(f"LINE: {line_number}")
        print(f"BYTE: {data[pos:pos+1]!r} = 0x{data[pos]:02X}")

        start = max(0, pos - 100)
        end = min(len(data), pos + 100)

        print("\nCONTEXT:")
        print(data[start:end].decode("utf-8", errors="replace"))

## Step 3: Review Common Adverbs

Stage 1 has written `common_adverbs.txt`. Review and edit it if needed before
proceeding to Stage 2. The adverb list is used both for masking and to protect
common adverbs from -que splitting.

In [20]:
print(
    f"\nReview {config.common_adverbs_path} and edit if needed, then proceed to Stage 2."
)


Review common_adverbs.txt and edit if needed, then proceed to Stage 2.


## Step 4: Run Stage 2 — -que Split, UDPipe, Mask

This runs the full stage 2 pipeline per file:
1. Read sentence-split files from Stage 1
2. Load common adverbs; add -que adverbs to effective blacklist
3. Apply -que splitting → write `*.quesplit.txt`
4. UDPipe POS tagging (with caching)
5. Two-pass masking → write `*.quesplit.masked.txt`

In [22]:
stage2_result = run_pipeline_stage2(txt_files, config=config)

print(f"\n=== Stage 2 Summary ===")
print(f"Output files: {len(stage2_result.output_files)}")
print(f"Sentences processed: {stage2_result.sentences_processed}")
print(f"Cache hits: {stage2_result.cache_hits}")
for f in stage2_result.output_files:
    print(f"  → {f.name}")

→ AE_bhc.txt
✓ AE_bhc.txt: 117 -que splits (fetched)
    399 masked sentences -> AE_bhc_sentences.quesplit.masked.txt
→ ASt_troilus.txt
✓ ASt_troilus.txt: 625 -que splits (fetched)
    3497 masked sentences -> ASt_troilus_sentences.quesplit.masked.txt
→ BB_adelae.txt
✓ BB_adelae.txt: 243 -que splits (fetched)
    700 masked sentences -> BB_adelae_sentences.quesplit.masked.txt
→ BBi_speculum.txt
✓ BBi_speculum.txt: 306 -que splits (fetched)
    981 masked sentences -> BBi_speculum_sentences.quesplit.masked.txt
→ BI_reynardus.txt
✓ BI_reynardus.txt: 404 -que splits (fetched)
    764 masked sentences -> BI_reynardus_sentences.quesplit.masked.txt
→ BM_regum.txt
✓ BM_regum.txt: 216 -que splits (fetched)
    610 masked sentences -> BM_regum_sentences.quesplit.masked.txt
→ BS_mathematicus.txt
✓ BS_mathematicus.txt: 229 -que splits (fetched)
    403 masked sentences -> BS_mathematicus_sentences.quesplit.masked.txt
→ EM_mahumeti.txt
✓ EM_mahumeti.txt: 173 -que splits (fetched)
    530 masked se

## Step 5: View Results

Compare original, sentences, and masked output for the first file.

In [24]:
# Show example output
example_file = txt_files[0]  # First input file
print(f"=== Original ({example_file.name}) ===")
with open(example_file, "r", encoding="utf-8") as f:
    original = f.read()
print(original[:1000] + "..." if len(original) > 1000 else original)

sentences_file = config.output_dir / f"{example_file.stem}_sentences.txt"
print(f"\n=== Sentences ({sentences_file.name}) ===")
with open(sentences_file, "r", encoding="utf-8") as f:
    sentences = f.readlines()
for i, sent in enumerate(sentences[:10]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(sentences)} total sentences)")

masked_file = config.output_dir / f"{example_file.stem}_sentences.quesplit.masked.txt"
print(f"\n=== Masked ({masked_file.name}) ===")
with open(masked_file, "r", encoding="utf-8") as f:
    masked = f.readlines()
for i, sent in enumerate(masked[:10]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(masked)} total sentences)")

=== Original (AE_bhc.txt) ===
Ante dies omnes mundi fuit omnis in uno
	Machina momento facta iubente Deo.
Sed tunc nec celum, nec terra, nec unda, nec aer
	Ornatus habuit quos habet, unde nitet.
Vnda tegit terram, tegit aera, sic elementa
	Hec tria miscentur efficiuntque chaos.
Hec polus empireus superat, ternos ter in ista
	Angelicos cetus collocat arce Deus.
Lumine virtutum cunctis his angelus unus
	Prelucens, dictus Lucifer inde fuit.
Hunc tumor et multos a celo tendit ad ima;
	Qui fuerant humiles promeruere statum.
Nec possunt stantes labi, nec surgere lapsi:
	Pars nequit ista malum uelle, nec illa bonum.
Que Deus ante diem fecit, sex illa diebus
	Protrahit in species et speciosa facit.
His factis opifex requieuit, id est, opem
     Desiit et noua post non operatus erat
Facta die prima fuerat lux prima, secunda
	Fit firmamentum summa quod astra tenet.
Tercia lux faciem terre detexit ab undis,
	Herbis et quauis arbore ditat eam.
Quarta dies solem, lunam stellasque creauit,
	Quinta d